# Alura Agente — Agente IA sobre documentos (Torneo fisticio )NBA2K Tournament Hub

Este notebook construye un agente de IA con **LangChain + Gemini** capaz de responder preguntas en lenguaje natural sobre un documento (PDF o CSV).

Como fuente de información usamos el documento de **reglas y descripción del torneo NBA2K** creado previamente para el agente.

**Flujo:**
1. Instalar dependencias
2. Configurar la API Key de Gemini
3. Subir y leer el documento (PDF/CSV)
4. Dividir el texto en fragmentos (chunks)
5. Crear embeddings y una base vectorial (FAISS)
6. Construir la cadena de preguntas y respuestas (RetrievalQA)
7. Probar el agente con preguntas de ejemplo

## 1. Instalar dependencias

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters pypdf pandas faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Configurar la API Key de Gemini


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

## 3. Subir el documento



In [ ]:
from google.colab import files

uploaded = files.upload()
doc_path = list(uploaded.keys())[0]
print(f"Archivo cargado: {doc_path}")

Saving reglas_torneo.pdf to reglas_torneo (1).pdf
Saving descripcion_torneo.pdf to descripcion_torneo.pdf
Archivo cargado: reglas_torneo (1).pdf


## 4. Leer y procesar el documento (PDF o CSV)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

if doc_path.lower().endswith(".pdf"):
    loader = PyPDFLoader(doc_path)
else:
    loader = CSVLoader(doc_path, encoding="utf-8")

raw_docs = loader.load()
print(f"Páginas/registros cargados: {len(raw_docs)}")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(raw_docs)
print(f"Fragmentos (chunks) generados: {len(chunks)}")

/tmp/ipykernel_3665/3285227200.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader


Páginas/registros cargados: 2
Fragmentos (chunks) generados: 4


## 5. Crear embeddings y base vectorial (FAISS)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Guardamos el índice para poder reutilizarlo en el deploy (carpeta faiss_index)
vectorstore.save_local("faiss_index")
print("Base vectorial creada y guardada en 'faiss_index'.")

Base vectorial creada y guardada en 'faiss_index'.


## 6. Construir el agente

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)

# Prompt para permitir conocimiento general
prompt_template = """Eres un asistente experto en el torneo NBA2K y en baloncesto/cultura general.

Sigue estas reglas para responder:
1. Si la pregunta es sobre el torneo NBA2K (reglas, equipos, formato, horarios, etc.), responde usando PRIORITARIAMENTE la información del siguiente contexto.
2. Si la pregunta NO está relacionada con el torneo (por ejemplo, historia de la NBA, jugadores como Michael Jordan, reglas generales de baloncesto, etc.), responde utilizando tu conocimiento general de forma clara y amable.
3. Si la pregunta ES sobre el torneo pero la respuesta específica NO está en el contexto, di claramente que no aparece en el reglamento (NO inventes fechas, premios ni reglas del torneo).

Contexto del torneo:
{context}

Pregunta: {question}

Respuesta:"""

QA_PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | QA_PROMPT
    | llm
    | StrOutputParser()
)

print("Agente híbrido listo.")

Agente híbrido listo.


## 7. Probar el agente

Cambia las preguntas por las que tengan sentido para tu documento del torneo (equipos, formato de eliminación, fechas, reglas, premios, etc.).

In [ ]:
import time

preguntas = [
    "¿Cuántos equipos participan en el torneo?",
    "¿Cuál es el formato de eliminación del torneo?",
    "¿Qué reglas aplican para las series de playoffs?",
]

for p in preguntas:
    respuesta = qa_chain.invoke(p)
    print(f"P: {p}")
    print(f"R: {respuesta}\n")

    # Pausa de 2 a 5 segundos para respetar los límites del Free Tier
    time.sleep(4)

P: ¿Cuántos equipos participan en el torneo?
R: Basándome únicamente en el contexto proporcionado, **no se especifica** la cantidad exacta de equipos que participan en el torneo. 

Solo se menciona que hay al menos dos grupos (Grupo A y Grupo B) y que clasifican los dos primeros de cada grupo a las semifinales, pero el número total de participantes o equipos no aparece en el reglamento.

P: ¿Cuál es el formato de eliminación del torneo?
R: El formato de la fase de eliminación del torneo es el siguiente:

* **Semifinal 1:** 1ro del Grupo A vs. 2do del Grupo B.
* **Semifinal 2:** 1ro del Grupo B vs. 2do del Grupo A.
* **Gran Final:** Ganador de la SF1 vs. Ganador de la SF2.
* **Modalidad:** Todos los partidos de eliminación son a partido único.
* **Desempate:** En caso de empate al final del tiempo reglamentario, se juega un período de *overtime* (prórroga).

P: ¿Qué reglas aplican para las series de playoffs?
R: Según la sección de **Fase de Eliminación** del reglamento, no existen "ser

## 8. Preguntas libres (interactivo)

In [ ]:
while True:
    pregunta = input("Tu pregunta (o 'salir'): ")
    if pregunta.lower() == "salir":
        break
    respuesta = qa_chain.invoke(pregunta)
    print(f"R: {respuesta}\n")

Tu pregunta (o 'salir'): Quien es michael jordan?
R: **Michael Jordan** es ampliamente considerado como uno de los mejores jugadores de baloncesto de todos los tiempos y una de las figuras deportivas más influyentes de la historia.

Aquí te comparto algunos aspectos clave de su leyenda:

* **Trayectoria:** Desarrolló la mayor parte y lo más glorioso de su carrera con los **Chicago Bulls** (1984-1993 y 1995-1998), y tuvo un último regreso con los **Washington Wizards** (2001-2003).
* **Campeonatos:** Ganó **6 anillos de la NBA** con los Chicago Bulls, logrando dos "tripletes" o *three-peats* (tres campeonatos consecutivos: 1991-1993 y 1996-1998).
* **Premios y Logros:**
  * 5 veces MVP (Jugador Más Valioso) de la temporada regular.
  * 6 veces MVP de las Finales de la NBA (en las 6 finales que jugó y ganó).
  * 14 veces elegido para el All-Star Game.
  * 10 veces máximo anotador de la temporada en la NBA.
  * Medalla de oro olímpica con el famoso "Dream Team" de EE. UU. en Barcelona 199

## 9. Descargar el índice para el deploy en OCI

Descarga la carpeta `faiss_index` (comprimida) para usarla en `app.py` cuando despliegues en OCI Compute.

In [ ]:
import shutil
from google.colab import files as colab_files

shutil.make_archive("faiss_index", "zip", "faiss_index")
colab_files.download("faiss_index.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>